# Vector Search

This notebook demonstrates Databend's vector search capabilities:
- Creating tables with vector columns
- Inserting embedding vectors
- Similarity search with cosine distance

In [ ]:
from databend_driver import connect
import random

conn = connect("databend+local:///./local-state")

## 1. Create a table with vector column

In [ ]:
conn.execute("DROP TABLE IF EXISTS documents")
conn.execute("""
    CREATE TABLE documents (
        id INT,
        content VARCHAR,
        embedding FLOAT64 VECTOR(4)
    )
""")
print("Table with vector column created!")

## 2. Insert data with embeddings

In a real application, embeddings would come from a model like OpenAI or a local embedding model.

In [ ]:
# Sample documents with dummy 4-dimensional embeddings
documents = [
    (1, 'How to train a machine learning model', [0.8, 0.2, 0.1, 0.3]),
    (2, 'Best practices for SQL optimization', [0.1, 0.9, 0.2, 0.1]),
    (3, 'Introduction to deep learning', [0.7, 0.3, 0.2, 0.4]),
    (4, 'Data engineering with Python', [0.2, 0.1, 0.8, 0.3]),
    (5, 'Cloud computing fundamentals', [0.1, 0.2, 0.3, 0.9]),
]

for doc_id, content, embedding in documents:
    embedding_str = str(embedding)
    conn.execute(
        f"INSERT INTO documents VALUES ({doc_id}, '{content}', '{embedding_str}')"
    )

print(f"Inserted {len(documents)} documents")

## 3. Similarity search

Find documents most similar to a query embedding using cosine distance.

In [ ]:
# Query: "machine learning" - similar to documents 1 and 3
query_embedding = [0.75, 0.25, 0.15, 0.35]

result = conn.query("""
    SELECT id, content,
           cosine_distance(embedding, '{query_embedding}') AS distance
    FROM documents
    ORDER BY distance
    LIMIT 3
"""")
print("Top 3 similar documents:")
for row in result.fetchall():
    print(f"  ID: {row[0]}, Content: {row[1]}, Distance: {row[2]:.4f}")

## 4. Vector search with filtering

In [ ]:
# Combine vector search with SQL filters
result = conn.query("""
    SELECT id, content,
           cosine_distance(embedding, '[0.5, 0.5, 0.5, 0.5]') AS distance
    FROM documents
    WHERE id != 1
    ORDER BY distance
    LIMIT 3
""")
print("Filtered vector search results:")
for row in result.fetchall():
    print(f"  ID: {row[0]}, Content: {row[1]}, Distance: {row[2]:.4f}")